In [2]:
from pathlib import Path
import pandas as pd
import re


import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, precision_recall_curve, ConfusionMatrixDisplay, balanced_accuracy_score)
import joblib


import joblib, os
import numpy as np
import pandas as pd
import torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import shap
import phate
from scipy.stats import spearmanr

# Set reproducibility seeds
np.random.seed(42)
torch.manual_seed(42)
import random
random.seed(42)


In [3]:
# Load the raw tables
DATA_DIR = Path("/home/gandalf/Documents/Data/Deep Mut")
MUT_INFO_PATH     = DATA_DIR / 'mut_info.tsv'
RNASEQ_PATH = DATA_DIR / "rnaseq_data.tsv"
META_PATH = DATA_DIR / "meta_data.tsv"

# Load metadata normally
meta_df = pd.read_csv(META_PATH, sep="\t")

# Read first row for gene names
header_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()

# Read data rows (skip header)
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

# Pad/truncate to match header length
if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for i in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names

# Rename first column to "sample_id"
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f"RNA-seq shape: {rnaseq_df.shape}")

KeyboardInterrupt: 

In [4]:
sorted(meta_df["histological_grade"].dropna().unique())

['G1',
 'G2',
 'G3',
 'G4',
 'GB',
 'GX',
 'High Grade',
 'Low Grade',
 '[Discrepancy]',
 '[Unknown]']

In [5]:
TCGA_ID_PATTERN = re.compile(r"TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+")


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')

    
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')

    if "histological_grade" in meta.columns:
        meta["histological_grade"] = (
            meta["histological_grade"].astype(str).str.strip().str.upper().replace({"": pd.NA})
        )

    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    """
    RNA-seq data structure:
    - Header row: gene names (19311 fields)
    - Data rows: TCGA sample ID in col 1, then 19310 expression values
    """
    # Extract gene names from header
    gene_names = df.columns.astype(str).str.strip().str.replace('"', '')
    
    # Extract sample IDs from first column (no normalization)
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    
    # Extract expression data: columns 2 onwards
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gene_names[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors="coerce")
    
    # Create dataframe with sample_id and gene expressions
    rna_clean = pd.concat([
        sample_ids.reset_index(drop=True),
        expr_data.reset_index(drop=True)
    ], axis=1)
    
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    
    # Remove any empty sample IDs
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    
    rna_clean = rna_clean.set_index('patient_id')
    
    # Handle duplicate patients by averaging
    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T  # genes x patients
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    
    return rna_final


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

rnaseq_patient_ids = set(clean_rnaseq_df.columns) - {"gene_id"}
matched_patient_ids = sorted(set(clean_meta_df["patient_id"]) & rnaseq_patient_ids)

clean_meta_df = clean_meta_df[clean_meta_df["patient_id"].isin(matched_patient_ids)].reset_index(drop=True)
clean_rnaseq_df = clean_rnaseq_df[["gene_id"] + matched_patient_ids]

print(f"Matched {len(matched_patient_ids)} patients across both tables")
print(f"Metadata: {len(clean_meta_df)} rows")
print(f"RNA-seq: {len(clean_rnaseq_df)} genes × {len(matched_patient_ids)} samples")
clean_meta_df.head()

Matched 9626 patients across both tables
Metadata: 9626 rows
RNA-seq: 19310 genes × 9626 samples


,sample_type_id,sample_type,project_id,rnaseqid,mutid,sample,x_patient,cancer.type.abbreviation,age_at_initial_pathologic_diagnosis,gender,...,os.time,dss,dss.time,dfi,dfi.time,pfi,pfi.time,redaction,x_primary_disease,patient_id
0,1,Primary Tumor,TCGA-GBM,TCGA-02-0047-01A,TCGA-02-0047-01,TCGA-02-0047-01,TCGA-02-0047,GBM,78.0,MALE,...,448.0,1.0,448.0,NaN,NaN,1.0,57.0,NaN,glioblastoma multiforme,TCGA-02-0047-01
1,1,Primary Tumor,TCGA-GBM,TCGA-02-0055-01A,TCGA-02-0055-01,TCGA-02-0055-01,TCGA-02-0055,GBM,62.0,FEMALE,...,76.0,1.0,76.0,NaN,NaN,1.0,6.0,NaN,glioblastoma multiforme,TCGA-02-0055-01
2,1,Primary Tumor,TCGA-GBM,TCGA-02-2483-01A,TCGA-02-2483-01,TCGA-02-2483-01,TCGA-02-2483,GBM,43.0,MALE,...,466.0,0.0,466.0,NaN,NaN,0.0,466.0,NaN,glioblastoma multiforme,TCGA-02-2483-01
3,1,Primary Tumor,TCGA-GBM,TCGA-02-2485-01A,TCGA-02-2485-01,TCGA-02-2485-01,TCGA-02-2485,GBM,53.0,MALE,...,470.0,0.0,470.0,NaN,NaN,1.0,186.0,NaN,glioblastoma multiforme,TCGA-02-2485-01
4,1,Primary Tumor,TCGA-GBM,TCGA-02-2486-01A,TCGA-02-2486-01,TCGA-02-2486-01,TCGA-02-2486,GBM,64.0,MALE,...,618.0,1.0,618.0,NaN,NaN,1.0,618.0,NaN,glioblastoma multiforme,TCGA-02-2486-01


In [6]:
meta_test = meta_df.copy()
cols_lower = (
    meta_test.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
meta_test.columns = cols_lower

id_candidates = ['sample', 'x_patient', 'rnaseqid', 'sample_type_id', 'mutid']

def _normalize_patient_id(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('"', '')
    return s if s else pd.NA

def _select_patient_id_test(row):
    for col in id_candidates:
        val = row.get(col)
        normalized = _normalize_patient_id(val)
        print(f"    {col}: {val} -> {normalized}")
        if not pd.isna(normalized):
            return normalized
    return pd.NA

In [7]:
print("Meta patient_id):")
print(clean_meta_df['patient_id'].head().tolist())

print("\nRNA-seq patient_id:")
print(list(rnaseq_patient_ids)[:5])

print(f"\nMeta unique count: {len(set(clean_meta_df['patient_id']))}")
print(f"RNA-seq unique count: {len(rnaseq_patient_ids)}")

print("\nIntersection:")
intersection = set(clean_meta_df["patient_id"]) & rnaseq_patient_ids
print(f"Matched: {len(intersection)}")

Meta patient_id):
['TCGA-02-0047-01', 'TCGA-02-0055-01', 'TCGA-02-2483-01', 'TCGA-02-2485-01', 'TCGA-02-2486-01']

RNA-seq patient_id:
['TCGA-VQ-A8PH-01', 'TCGA-YU-A94D-01', 'TCGA-2Z-A9JN-01', 'TCGA-AA-3696-01', 'TCGA-KN-8437-01']

Meta unique count: 9626
RNA-seq unique count: 9626

Intersection:
Matched: 9626


In [8]:
print("=== DATASET BEFORE LABELING ===")
print(f"Metadata shape: {clean_meta_df.shape}")
print(f"RNA-seq shape: {clean_rnaseq_df.shape}")

# Map cancer grades to binary risk labels
# low risk: G1 / G2/ LOW GRADE
# high risk: G3 / G4 / HIGH GRADE
low_risk_grades = {'G1', 'LOW GRADE'}
high_risk_grades = {'G3', 'G4', 'HIGH GRADE'}

def grade_to_risk_label(grade):
    """Map histological grade to binary risk label."""
    if pd.isna(grade):
        return pd.NA
    grade_str = str(grade).strip().upper()
    if grade_str in low_risk_grades:
        return 0  # Low risk
    elif grade_str in high_risk_grades:
        return 1  # High risk
    else:
        return pd.NA  # Unknown

clean_meta_df['risk_label'] = clean_meta_df['histological_grade'].apply(grade_to_risk_label)

# Filter to only samples with valid risk labels
valid_idx = clean_meta_df['risk_label'].notna()
labeled_meta_df = clean_meta_df[valid_idx].reset_index(drop=True)

# Filter RNA-seq to match
valid_patient_ids = set(labeled_meta_df['patient_id'])
labeled_rnaseq_df = clean_rnaseq_df[['gene_id'] + sorted([p for p in clean_rnaseq_df.columns if p in valid_patient_ids])].reset_index(drop=True)

print("\n=== DATASET AFTER LABELING ===")
print(f"Metadata shape: {labeled_meta_df.shape}")
print(f"RNA-seq shape: {labeled_rnaseq_df.shape}")
print(f"\nRisk label distribution:")
print(labeled_meta_df['risk_label'].value_counts().sort_index())
print(f"  0 (Low risk):  {(labeled_meta_df['risk_label'] == 0).sum()}")
print(f"  1 (High risk): {(labeled_meta_df['risk_label'] == 1).sum()}")

=== DATASET BEFORE LABELING ===
Metadata shape: (9626, 41)
RNA-seq shape: (19310, 9627)

=== DATASET AFTER LABELING ===
Metadata shape: (2494, 42)
RNA-seq shape: (19310, 2495)

Risk label distribution:
risk_label
0     312
1    2182
Name: count, dtype: int64
  0 (Low risk):  312
  1 (High risk): 2182



### Gene Expression-Based Cancer Grade Prediction Pipeline

### Using Supervised PHATE + Neural Feature Selector + XGBoost


#### =============================================================================
#### STAGE 0: Data Preparation
#### =============================================================================

In [9]:


def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42, output_dir='deployment'):

    os.makedirs(output_dir, exist_ok=True)
    
    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))
    
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]
    
    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))
    
    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]
    
    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler
    scaler_path = os.path.join(output_dir, 'scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'scaler_path': scaler_path,
        'n_features': X.shape[1],
    }
    
    print(f"STAGE 0: Data Prepared")
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print(f"  Label distribution:")
    print(f"    Train - Low: {(y_train==0).sum()}, High: {(y_train==1).sum()}")
    print(f"    Val   - Low: {(y_val==0).sum()}, High: {(y_val==1).sum()}")
    print(f"    Test  - Low: {(y_test==0).sum()}, High: {(y_test==1).sum()}")
    
    return result

X_raw = labeled_rnaseq_df.drop('gene_id', axis=1).T.values  # (samples x genes)
y_binary = labeled_meta_df['risk_label'].values

data = stage_0_prepare_data(X_raw, y_binary, output_dir='deployment')


STAGE 0: Data Prepared
  Train: (1745, 19310) | Val: (374, 19310) | Test: (375, 19310)
  Label distribution:
    Train - Low: 218, High: 1527
    Val   - Low: 47, High: 327
    Test  - Low: 47, High: 328


#### =============================================================================
#### STAGE 1: Unsupervised PHATE
#### =============================================================================

In [ ]:
def stage_1_unsupervised_phate(X_train, y_train, X_val, y_val, 
                              n_components=100, 
                              output_dir='deployment'):
    
    print(f"  Training PHATE (unsupervised) with {n_components} components")
    print(f"  X_train shape: {X_train.shape}")
    
    # Fit PHATE on gene expression only (no labels)
    phate_op = phate.PHATE(
        n_components=n_components,
        random_state=42,
        knn=5,
        decay=40,
        n_jobs=-1,
        verbose=0,
        mds_solver="smacof",
    )
    t0 = time.time()
    X_train_phate = phate_op.fit_transform(X_train)
    phate_train_time = time.time() - t0
    
    # Transform validation set
    t0 = time.time()
    X_val_phate = phate_op.transform(X_val)
    phate_infer_per_sample_ms = (time.time() - t0) / max(len(X_val), 1) * 1000
    
    # Save PHATE model
    phate_path = os.path.join(output_dir, 'phate.pkl')
    joblib.dump(phate_op, phate_path)
    
    config = {
        'supervision_weight': 0.0,
        'n_components': n_components,
        'supervised': False
    }
    config_path = os.path.join(output_dir, 'phate_config.json')
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
   
    result = {
        'phate_op': phate_op,
        'X_train_phate': X_train_phate,
        'X_val_phate': X_val_phate,
        'supervision_weight': 0.0,
        'phate_path': phate_path,
        'config_path': config_path,
        'train_time_s': phate_train_time,
        'infer_time_per_sample_ms': phate_infer_per_sample_ms,
    }
    
    print(f"STAGE 1: Unsupervised PHATE Fitted")
    print(f"  X_train_phate: {X_train_phate.shape}")
    print(f"  X_val_phate: {X_val_phate.shape}")
    print(f"  Train time: {phate_train_time:.2f}s")
    print(f"  Infer time/sample (val): {phate_infer_per_sample_ms:.3f}ms")
    
    return result

# Execute Stage 1
phate_result = stage_1_unsupervised_phate(
    data['X_train'], data['y_train'],
    data['X_val'], data['y_val'],
    n_components=100,
    output_dir='deployment'
)


  Training PHATE (unsupervised) with 100 components
  X_train shape: (1745, 19310)
STAGE 1: Unsupervised PHATE Fitted
  X_train_phate: (1745, 100)
  X_val_phate: (374, 100)


#### =============================================================================
#### STAGE 2: SHAP Ranking on PHATE Components
#### =============================================================================

In [ ]:


def stage_2_shap_ranking(X_train_phate, y_train, X_val_phate, y_val,
                        top_k=30, output_dir='deployment'):
    """
    Train XGBoost on PHATE components, use SHAP to rank importance.
    """
    
    print(f"  Training XGBoost on {X_train_phate.shape[1]} PHATE components...")
    
    # Train XGBoost without early stopping for simplicity, validate manually
    xgb_ranking = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=20,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbose=0
    )
    
    t0 = time.time()
    xgb_ranking.fit(
        X_train_phate, y_train
    )
    xgb_rank_train_time = time.time() - t0
    
    # Compute SHAP values on validation set
    print(f"  Computing SHAP values for ranking...")
    explainer = shap.TreeExplainer(xgb_ranking)
    t0 = time.time()
    shap_values = explainer.shap_values(X_val_phate)
    shap_time = time.time() - t0
    
    # Handle binary classification (shap returns list of 2 arrays)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Use positive class
    
    # Compute mean absolute SHAP per component
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Rank components
    ranked_indices = np.argsort(mean_abs_shap)[::-1]
    top_k_indices = ranked_indices[:top_k].tolist()
    
    print(f"  Top-{top_k} components (by mean SHAP):")
    for i, idx in enumerate(top_k_indices[:5]):
        print(f"    {i+1}. Component {idx}: {mean_abs_shap[idx]:.4f}")
    
    # Save top-K indices
    topk_path = os.path.join(output_dir, 'top_k_components.json')
    with open(topk_path, 'w') as f:
        json.dump({
            'top_k': top_k,
            'component_indices': top_k_indices,
            'mean_abs_shap_values': mean_abs_shap.tolist()
        }, f, indent=2)
    
    # Plot SHAP summary (bar plot for top-K)
    fig, ax = plt.subplots(figsize=(10, 6))
    top_indices_for_plot = ranked_indices[:20]
    shap_vals_for_plot = mean_abs_shap[top_indices_for_plot]
    ax.barh(range(len(top_indices_for_plot)), shap_vals_for_plot)
    ax.set_yticks(range(len(top_indices_for_plot)))
    ax.set_yticklabels([f'Component {i}' for i in top_indices_for_plot])
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('SHAP Ranking: Top 20 PHATE Components')
    ax.invert_yaxis()
    plt.tight_layout()
    shap_plot_path = os.path.join(output_dir, 'shap_ranking_bar.png')
    plt.savefig(shap_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    result = {
        'xgb_ranking': xgb_ranking,
        'top_k_indices': top_k_indices,
        'mean_abs_shap': mean_abs_shap,
        'shap_values_val': shap_values,
        'topk_path': topk_path,
        'xgb_rank_train_time_s': xgb_rank_train_time,
        'shap_time_s': shap_time,
    }
    
    print(f" STAGE 2: SHAP Ranking Complete")
    print(f"  Top-K components saved to: {topk_path}")
    print(f"  SHAP summary plot saved to: {shap_plot_path}")
    print(f"  XGBoost ranking train time: {xgb_rank_train_time:.2f}s")
    print(f"  SHAP computation time: {shap_time:.2f}s")
    
    return result

# Execute Stage 2
shap_result = stage_2_shap_ranking(
    phate_result['X_train_phate'], data['y_train'],
    phate_result['X_val_phate'], data['y_val'],
    top_k=30,
    output_dir='deployment'
)



  Training XGBoost on 100 PHATE components...
  Computing SHAP values for ranking...
  Top-30 components (by mean SHAP):
    1. Component 6: 0.7265
    2. Component 1: 0.6357
    3. Component 41: 0.3070
    4. Component 15: 0.2793
    5. Component 7: 0.2779
 STAGE 2: SHAP Ranking Complete
  Top-K components saved to: deployment/top_k_components.json
  SHAP summary plot saved to: deployment/shap_ranking_bar.png


#### =============================================================================
#### STAGE 3: Train MLP Feature Selector
#### =============================================================================

In [ ]:
class FeatureSelectorMLP(nn.Module):
    def __init__(self, n_input_genes, n_output_components, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [1024, 512, 256]

        layers = []
        prev_dim = n_input_genes

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, n_output_components))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def stage_3_train_component_predictor(X_train, X_val, X_train_phate, X_val_phate,
                                      top_k_indices, output_dir='deployment',
                                      epochs=100, batch_size=32, lr=1e-3, patience=10,
                                      y_train_labels=None):
    """
    Train MLP to predict top-K PHATE component values directly from raw gene features.
    If labels are provided, uses class-balanced sampling in training batches.
    """

    from torch.utils.data import WeightedRandomSampler

    component_targets_train = X_train_phate[:, top_k_indices]
    component_targets_val = X_val_phate[:, top_k_indices]

    print(f"  Component targets shape (train): {component_targets_train.shape}")
    print(f"  Component targets shape (val):   {component_targets_val.shape}")

    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(component_targets_train)
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    if y_train_labels is not None:
        y_train_labels = np.asarray(y_train_labels).astype(int)
        class_counts = np.bincount(y_train_labels)
        class_weights = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights[y_train_labels]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
        print(f"  Using balanced sampler in Stage 3: class_counts={class_counts.tolist()}")
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        print("  Using standard shuffled sampling in Stage 3")

    X_val_tensor = torch.FloatTensor(X_val)
    y_val_tensor = torch.FloatTensor(component_targets_val)

    hidden_dims = [1024, 512, 256]
    model = FeatureSelectorMLP(
        n_input_genes=X_train.shape[1],
        n_output_components=len(top_k_indices),
        hidden_dims=hidden_dims,
    )
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    print(f"  Training component predictor MLP for {epochs} epochs...")
    t_train = time.time()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_tensor)
            val_loss = criterion(val_pred, y_val_tensor).item()
        val_losses.append(val_loss)
        model.train()

        scheduler.step(val_loss)

        if (epoch + 1) % 20 == 0:
            print(f"    Epoch {epoch+1}: train_loss={epoch_loss:.4f}, val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    mlp_train_time = time.time() - t_train

    model_path = os.path.join(output_dir, 'component_predictor_mlp.pt')
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'n_input_genes': int(X_train.shape[1]),
        'n_output_components': int(len(top_k_indices)),
        'hidden_dims': hidden_dims,
        'top_k_indices': [int(i) for i in top_k_indices],
    }
    torch.save(checkpoint, model_path)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_losses, label='Train MSE', linewidth=2)
    ax.plot(val_losses, label='Val MSE', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('MLP Component Predictor: Training Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    loss_plot_path = os.path.join(output_dir, 'mlp_component_predictor_training_curve.png')
    plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
    plt.close()

    result = {
        'model': model,
        'model_path': model_path,
        'target_train_components': component_targets_train,
        'target_val_components': component_targets_val,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_mse': best_val_loss,
        'train_time_s': mlp_train_time,
    }

    print(f" STAGE 3: MLP Component Predictor Trained")
    print(f"  Best val MSE: {best_val_loss:.4f}")
    print(f"  Train time: {mlp_train_time:.2f}s")

    return result


mlp_result = stage_3_train_component_predictor(
    data['X_train'], data['X_val'],
    phate_result['X_train_phate'], phate_result['X_val_phate'],
    shap_result['top_k_indices'],
    output_dir='deployment',
    epochs=100, batch_size=32, lr=1e-3, patience=10,
    y_train_labels=data['y_train']
)


  Component targets shape (train): (1745, 30)
  Component targets shape (val):   (374, 30)
  Using balanced sampler in Stage 3: class_counts=[218, 1527]
  Training component predictor MLP for 100 epochs...
    Epoch 20: train_loss=0.0001, val_loss=0.0000
    Epoch 40: train_loss=0.0000, val_loss=0.0000
    Early stopping at epoch 43
 STAGE 3: MLP Component Predictor Trained
  Best val MSE: 0.0000


#### =============================================================================
#### STAGE 3b: Validate Selector
##### =============================================================================

In [13]:
def stage_3b_validate_selector(X_test, y_test, model,
                               top_k_indices, phate_op):
    """
    Validate MLP by comparing predicted top-K components against true PHATE top-K components.
    """

    print("Applying MLP to test set...")

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        pred_components_test = model(X_test_tensor).numpy()

    # Build true PHATE components (unsupervised - no labels used)
    X_test_phate = phate_op.transform(X_test)
    true_components_test = X_test_phate[:, top_k_indices]

    mse = float(np.mean((pred_components_test - true_components_test) ** 2))

    # Rank consistency using average absolute component magnitudes
    mean_true = np.mean(np.abs(true_components_test), axis=0)
    mean_pred = np.mean(np.abs(pred_components_test), axis=0)
    true_rank = np.argsort(-mean_true)
    pred_rank = np.argsort(-mean_pred)
    corrcoef, pval = spearmanr(true_rank, pred_rank)

    print(f" STAGE 3b: Component Predictor Validation")
    print(f"  Test component MSE: {mse:.4f}")
    print(f"  Spearman r (true rank vs predicted rank): {corrcoef:.4f} (p={pval:.4e})")

    return {
        'component_mse': mse,
        'spearman_r': corrcoef,
        'spearman_p': pval,
        'pred_test_components': pred_components_test,
        'true_test_components': true_components_test,
    }


validation_result = stage_3b_validate_selector(
    data['X_test'], data['y_test'],
    mlp_result['model'],
    shap_result['top_k_indices'],
    phate_result['phate_op']
)

Applying MLP to test set...
 STAGE 3b: Component Predictor Validation
  Test component MSE: 0.0000
  Spearman r (true rank vs predicted rank): 0.4461 (p=1.3489e-02)


#### =============================================================================
#### STAGE 4: Final XGBoost Predictor
#### =============================================================================

In [14]:
# Transform test set through PHATE before Stage 4

X_test_phate = phate_result['phate_op'].transform(data['X_test'])
print(f" X_test_phate shape: {X_test_phate.shape}")
print(f" y_test dtype: {data['y_test'].dtype}, unique values: {np.unique(data['y_test'])}")

 X_test_phate shape: (375, 100)
 y_test dtype: object, unique values: [0 1]


In [ ]:
def stage_4_final_xgboost(X_train_scaled, y_train, X_val_scaled, y_val,
                          X_test_scaled, y_test, mlp_model,
                          top_k_indices, output_dir='deployment'):
    """
    Train final XGBoost on MLP-predicted top-K component representations.
    """
    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)

    t0 = time.time()
    mlp_model.eval()
    with torch.no_grad():
        X_train_topk = mlp_model(torch.FloatTensor(X_train_scaled)).numpy()
        X_val_topk = mlp_model(torch.FloatTensor(X_val_scaled)).numpy()
        X_test_topk = mlp_model(torch.FloatTensor(X_test_scaled)).numpy()
    n_total = len(X_train_scaled) + len(X_val_scaled) + len(X_test_scaled)
    mlp_infer_per_sample_ms = (time.time() - t0) / max(n_total, 1) * 1000

    print(f"  Using MLP-predicted top-{len(top_k_indices)} components")
    print(f"  Train feature matrix: {X_train_topk.shape}")

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    scale_pos_weight = float(n_neg) / float(max(n_pos, 1))
    n_samples = len(y_train)
    w_neg = n_samples / (2.0 * max(n_neg, 1))
    w_pos = n_samples / (2.0 * max(n_pos, 1))
    sample_weight = np.where(y_train == 1, w_pos, w_neg)

    print(f"  Class balance (train): neg={n_neg}, pos={n_pos}")
    print(f"  Imbalance settings: scale_pos_weight={scale_pos_weight:.4f}, w_neg={w_neg:.4f}, w_pos={w_pos:.4f}")

    xgb_final = xgb.XGBClassifier(
        n_estimators=900,
        max_depth=20,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        max_delta_step=3,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbose=0
    )

    t0 = time.time()
    xgb_final.fit(X_train_topk, y_train, sample_weight=sample_weight)
    xgb_train_time = time.time() - t0

    t0 = time.time()
    y_pred_test = xgb_final.predict(X_test_topk)
    y_pred_proba_test = xgb_final.predict_proba(X_test_topk)
    xgb_infer_per_sample_ms = (time.time() - t0) / max(len(X_test_topk), 1) * 1000

    accuracy = accuracy_score(y_test, y_pred_test)
    balanced_acc = balanced_accuracy_score(y_test, y_pred_test)
    f1_weighted = f1_score(y_test, y_pred_test, average='weighted')
    cm = confusion_matrix(y_test, y_pred_test)

    print(f"  Test Accuracy: {accuracy:.4f}")
    print(f"  Test Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  Test F1 (weighted): {f1_weighted:.4f}")
    print(f"  XGBoost train time: {xgb_train_time:.2f}s")
    print(f"  MLP infer time/sample: {mlp_infer_per_sample_ms:.3f}ms")
    print(f"  XGBoost infer time/sample: {xgb_infer_per_sample_ms:.3f}ms")

    cm_labels = sorted(np.unique(y_test))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(f'Confusion Matrix\nAccuracy: {accuracy:.4f} | Bal Acc: {balanced_acc:.4f} | F1: {f1_weighted:.4f}',
                 fontsize=12, pad=12)
    plt.tight_layout()
    cm_path = os.path.join(output_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()

    explainer_final = shap.TreeExplainer(xgb_final)
    shap_values_final = explainer_final.shap_values(X_test_topk)
    if isinstance(shap_values_final, list):
        shap_values_final = shap_values_final[1]

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(
        shap_values_final,
        X_test_topk,
        feature_names=[f'PredComp {top_k_indices[i]}' for i in range(len(top_k_indices))],
        plot_type='bar',
        show=False
    )
    plt.tight_layout()
    shap_beeswarm_path = os.path.join(output_dir, 'shap_final_summary_bar.png')
    plt.savefig(shap_beeswarm_path, dpi=150, bbox_inches='tight')
    plt.close()

    model_path = os.path.join(output_dir, 'xgboost_grade_predictor.pkl')
    joblib.dump(xgb_final, model_path)

    result = {
        'xgb_final': xgb_final,
        'X_train_topk': X_train_topk,
        'X_val_topk': X_val_topk,
        'X_test_topk': X_test_topk,
        'y_pred_test': y_pred_test,
        'y_pred_proba_test': y_pred_proba_test,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'scale_pos_weight': scale_pos_weight,
        'class_weight_neg': float(w_neg),
        'class_weight_pos': float(w_pos),
        'shap_values': shap_values_final,
        'model_path': model_path,
        'cm_path': cm_path,
        'xgb_train_time_s': xgb_train_time,
        'mlp_infer_per_sample_ms': mlp_infer_per_sample_ms,
        'xgb_infer_per_sample_ms': xgb_infer_per_sample_ms,
    }

    return result


xgb_final_result = stage_4_final_xgboost(
    data['X_train'], data['y_train'],
    data['X_val'], data['y_val'],
    data['X_test'], data['y_test'],
    mlp_result['model'],
    shap_result['top_k_indices'],
    output_dir='deployment'
)


  Using MLP-predicted top-30 components
  Train feature matrix: (1745, 30)
  Class balance (train): neg=218, pos=1527
  Imbalance settings: scale_pos_weight=0.1428, w_neg=4.0023, w_pos=0.5714
  Test Accuracy: 0.8587
  Test Balanced Accuracy: 0.7278
  Test F1 (weighted): 0.8649


#### =============================================================================
#### STAGE 5: Gene Mapping
#### =============================================================================

In [16]:

def stage_5_gene_mapping(X_train_raw, X_train_phate, top_k_indices, 
                         gene_names, output_dir='deployment'):
    """
    For each top-K component, train LinearRegression: raw genes -> component value.
    Extract top-10 genes per component by absolute coefficient.
    """
    
    print(f"  Training LinearRegression for each of {len(top_k_indices)} components...")
    print(f"  X_train_raw shape: {X_train_raw.shape}, gene_names: {len(gene_names)}")
    
    component_gene_mapping = {}
    
    # Ensure gene_names length matches X_train_raw features
    if len(gene_names) != X_train_raw.shape[1]:
        raise ValueError(
            f"gene_names length ({len(gene_names)}) must match number of features ({X_train_raw.shape[1]})"
        )
    
    for i, component_idx in enumerate(top_k_indices):
        
        # Get component values
        y_component = X_train_phate[:, component_idx]
        
        # Fit LinearRegression
        lr = LinearRegression()
        lr.fit(X_train_raw, y_component)
        
        # Get absolute coefficients
        abs_coefs = np.abs(lr.coef_)
        
        # Top-10 genes by absolute coefficient
        top_10_indices = np.argsort(abs_coefs)[-10:][::-1]
        
        top_genes = [
            {
                'gene_name': str(gene_names[int(gene_idx)]),
                'coefficient': float(lr.coef_[gene_idx]),
                'abs_coefficient': float(abs_coefs[gene_idx])
            }
            for gene_idx in top_10_indices
        ]
        
        component_gene_mapping[f'component_{component_idx}'] = {
            'intercept': float(lr.intercept_),
            'top_10_genes': top_genes
        }
        
        if (i + 1) % 5 == 0:
            print(f"    Processed {i+1}/{len(top_k_indices)} components...")
    
    mapping_path = os.path.join(output_dir, 'component_gene_mapping.json')
    with open(mapping_path, 'w') as f:
        json.dump(component_gene_mapping, f, indent=2)
    
    print(f" STAGE 5: Gene Mapping Complete")
    print(f"  Mapping saved to: {mapping_path}")
    
    return component_gene_mapping

# Extract real gene IDs (rows in RNA-seq matrix)
gene_names = clean_rnaseq_df['gene_id'].tolist()

# Get raw training data by inverse-scaling the scaled data
X_train_raw = data['scaler'].inverse_transform(data['X_train'])

gene_mapping = stage_5_gene_mapping(
    X_train_raw,
    phate_result['X_train_phate'],
    shap_result['top_k_indices'],
    gene_names,
    output_dir='deployment'
)

print("Example mapping for first component:")
first_component = f"component_{shap_result['top_k_indices'][0]}"
print(json.dumps({first_component: gene_mapping[first_component]}, indent=2))


  Training LinearRegression for each of 30 components...
  X_train_raw shape: (1745, 19310), gene_names: 19310
    Processed 5/30 components...
    Processed 10/30 components...
    Processed 15/30 components...
    Processed 20/30 components...
    Processed 25/30 components...
    Processed 30/30 components...
 STAGE 5: Gene Mapping Complete
  Mapping saved to: deployment/component_gene_mapping.json
Example mapping for first component:
{
  "component_6": {
    "intercept": 0.00864927363739462,
    "top_10_genes": [
      {
        "gene_name": "NXPE2",
        "coefficient": 2.453117205047994e-05,
        "abs_coefficient": 2.453117205047994e-05
      },
      {
        "gene_name": "EEF1G",
        "coefficient": -2.3895643739231657e-05,
        "abs_coefficient": 2.3895643739231657e-05
      },
      {
        "gene_name": "AVPR1B",
        "coefficient": 2.277598183423014e-05,
        "abs_coefficient": 2.277598183423014e-05
      },
      {
        "gene_name": "KRT4",
        "c

#### =============================================================================
#### STAGE 5b: Downstream task: TP53 Mutation prediction
#### =============================================================================

In [ ]:
def stage_5b_tp53_prediction_XGBoost(data, y_binary, patient_ids_order, mlp_model,
                             mut_info_path, output_dir='deployment',
                             random_state=42, test_size=0.15, val_size=0.15):
    """
    Train TP53 mutation predictor on MLP-predicted top-K components.
    Binarization: wild -> 0, anything else -> 1.
    Classifier: XGBoost + threshold optimization on validation F1.
    """

    print(f"  Loading mutation table: {mut_info_path}")
    mut_df = pd.read_csv(mut_info_path, sep='\t', dtype=str)

    mut_df.columns = (
        mut_df.columns.astype(str)
        .str.strip().str.strip('"').str.lower()
        .str.replace(' ', '_', regex=False)
    )

    index_ids = pd.Index(mut_df.index).astype(str)
    index_tcga_fraction = index_ids.str.match(r'^TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+').mean()

    if index_tcga_fraction > 0.5:
        mut_df = mut_df.copy()
        mut_df['patient_id'] = index_ids.str.strip().str.replace('"', '')
        id_col = 'patient_id'
    else:
        id_candidates = ['patient_id', 'sample', 'x_patient', 'rnaseqid', 'sample_id', 'mutid']
        id_col = next((c for c in id_candidates if c in mut_df.columns), mut_df.columns[0])
        mut_df[id_col] = mut_df[id_col].astype(str).str.strip().str.replace('"', '')

    mut_df = mut_df.dropna(subset=[id_col]).drop_duplicates(id_col)

    mutation_cols = [c for c in mut_df.columns if c != id_col]
    tp53_col = next((c for c in mutation_cols if 'tp53' in c.lower()), None)
    if tp53_col is None:
        raise ValueError(f'TP53 column not found. Available: {mutation_cols[:10]}')
    print(f"  Found TP53 column: '{tp53_col}'")

    binary_mut_info = mut_df[[id_col, tp53_col]].copy()
    binary_mut_info[tp53_col] = binary_mut_info[tp53_col].apply(
        lambda x: 0 if str(x).strip().lower() == 'wild' else 1
    )
    mut_by_patient = binary_mut_info.set_index(id_col).reindex(patient_ids_order).fillna(0).astype(int)
    y_tp53_all = mut_by_patient[tp53_col].values

    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(np.zeros((len(y_binary), 1)), y_binary))
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx_rel, val_idx_rel = next(splitter2.split(np.zeros((len(train_val_idx), 1)), y_binary[train_val_idx]))
    train_idx = train_val_idx[train_idx_rel]
    val_idx   = train_val_idx[val_idx_rel]

    mlp_model.eval()
    with torch.no_grad():
        X_train_comp = mlp_model(torch.FloatTensor(data['X_train'])).numpy()
        X_val_comp   = mlp_model(torch.FloatTensor(data['X_val'])).numpy()
        X_test_comp  = mlp_model(torch.FloatTensor(data['X_test'])).numpy()

    y_train_tp53 = y_tp53_all[train_idx]
    y_val_tp53   = y_tp53_all[val_idx]
    y_test_tp53  = y_tp53_all[test_idx]

    train_pos = y_train_tp53.sum()
    train_neg = len(y_train_tp53) - train_pos
    print(f"  Training set TP53: {train_pos} mutated, {train_neg} wild-type")

    scale_pos_weight = float(train_neg) / float(max(train_pos, 1))
    print(f"  Using scale_pos_weight: {scale_pos_weight:.3f}")

    tp53_model = xgb.XGBClassifier(
        n_estimators=900, max_depth=20, learning_rate=0.07,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, max_delta_step=3,
        random_state=42, n_jobs=-1, eval_metric='logloss', verbose=0
    )

    t0 = time.time()
    tp53_model.fit(X_train_comp, y_train_tp53)
    tp53_train_time = time.time() - t0

    t0 = time.time()
    y_test_proba       = tp53_model.predict_proba(X_test_comp)[:, 1]
    y_test_pred_def    = tp53_model.predict(X_test_comp)
    tp53_infer_per_sample_ms = (time.time() - t0) / max(len(X_test_comp), 1) * 1000

    y_val_proba  = tp53_model.predict_proba(X_val_comp)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val_tp53, y_val_proba)
    f1_curve = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
    best_thresh = thresholds[np.argmax(f1_curve)]
    print(f"  Optimal decision threshold (val F1): {best_thresh:.4f}")

    y_test_pred_tuned = (y_test_proba >= best_thresh).astype(int)
    y_val_pred_def    = tp53_model.predict(X_val_comp)
    y_val_pred_tuned  = (y_val_proba >= best_thresh).astype(int)

    metrics = {}
    for split_name, y_true, y_def, y_tuned in [
        ('val',  y_val_tp53,  y_val_pred_def,  y_val_pred_tuned),
        ('test', y_test_tp53, y_test_pred_def, y_test_pred_tuned),
    ]:
        metrics[f'{split_name}_acc_def']    = accuracy_score(y_true, y_def)
        metrics[f'{split_name}_f1_def']     = f1_score(y_true, y_def)
        metrics[f'{split_name}_acc_tuned']  = accuracy_score(y_true, y_tuned)
        metrics[f'{split_name}_f1_tuned']   = f1_score(y_true, y_tuned)

    test_auc  = roc_auc_score(y_test_tp53, y_test_proba)
    cm_def    = confusion_matrix(y_test_tp53, y_test_pred_def)
    cm_tuned  = confusion_matrix(y_test_tp53, y_test_pred_tuned)

    model_path = os.path.join(output_dir, 'tp53_predictor_XGBoost.pkl')
    joblib.dump(tp53_model, model_path)

    print(f"\n STAGE 5b: TP53 Mutation Predictor (XGBoost)")
    print(f"  XGBoost train time: {tp53_train_time:.2f}s")
    print(f"  Infer time/sample:  {tp53_infer_per_sample_ms:.3f}ms")
    print(f"  Optimal threshold   : {best_thresh:.4f}")
    print(f"\n  ── Default threshold (0.50) ──")
    print(f"  Val  Acc: {metrics['val_acc_def']:.4f}  F1: {metrics['val_f1_def']:.4f}")
    print(f"  Test Acc: {metrics['test_acc_def']:.4f}  F1: {metrics['test_f1_def']:.4f}")
    print(f"  Confusion   [[TN={cm_def[0,0]}, FP={cm_def[0,1]}]  [FN={cm_def[1,0]}, TP={cm_def[1,1]}]]")
    print(f"\n  ── Tuned threshold ({best_thresh:.4f}) ──")
    print(f"  Val  Acc: {metrics['val_acc_tuned']:.4f}  F1: {metrics['val_f1_tuned']:.4f}")
    print(f"  Test Acc: {metrics['test_acc_tuned']:.4f}  F1: {metrics['test_f1_tuned']:.4f}")
    print(f"  Confusion   [[TN={cm_tuned[0,0]}, FP={cm_tuned[0,1]}]  [FN={cm_tuned[1,0]}, TP={cm_tuned[1,1]}]]")
    print(f"\n  Test AUC-ROC: {test_auc:.4f}")

    return {
        'tp53_model': tp53_model,
        'tp53_column': tp53_col,
        'scale_pos_weight': float(scale_pos_weight),
        'optimal_threshold': float(best_thresh),
        'test_auc': float(test_auc),
        'test_confusion_matrix': cm_def.tolist(),
        'test_confusion_matrix_tuned': cm_tuned.tolist(),
        'tp53_train_time_s': tp53_train_time,
        'tp53_infer_per_sample_ms': tp53_infer_per_sample_ms,
        'model_path': model_path,
        **{k: float(v) for k, v in metrics.items()},
    }


In [ ]:
patient_ids_order = labeled_rnaseq_df.columns.drop('gene_id').tolist()
MUT_INFO_PATH = DATA_DIR / 'mut_info.tsv'

tp53_result = stage_5b_tp53_prediction_XGBoost(
    data=data,
    y_binary=y_binary,
    patient_ids_order=patient_ids_order,
    mlp_model=mlp_result['model'],
    mut_info_path=MUT_INFO_PATH,
    output_dir='deployment',
)

print(f"\n  Test AUC-ROC       : {tp53_result['test_auc']:.4f}")
print(f"  Optimal threshold  : {tp53_result['optimal_threshold']:.4f}")
print(f"\n  [Default  threshold=0.50]")
print(f"  Val  Acc={tp53_result['val_acc_def']:.4f}  F1={tp53_result['val_f1_def']:.4f}")
print(f"  Test Acc={tp53_result['test_acc_def']:.4f}  F1={tp53_result['test_f1_def']:.4f}")
print(f"\n  [Tuned threshold={tp53_result['optimal_threshold']:.4f}]")
print(f"  Val  Acc={tp53_result['val_acc_tuned']:.4f}  F1={tp53_result['val_f1_tuned']:.4f}")
print(f"  Test Acc={tp53_result['test_acc_tuned']:.4f}  F1={tp53_result['test_f1_tuned']:.4f}")
cm = tp53_result['test_confusion_matrix_tuned']
print(f"  Confusion (tuned)  : TN={cm[0][0]}, FP={cm[0][1]}, FN={cm[1][0]}, TP={cm[1][1]}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  EXECUTION TIME SUMMARY
# ═══════════════════════════════════════════════════════════════
W = 55

print("=" * W)
print("  EXECUTION TIME SUMMARY  (GeneSieve pipeline)")
print("=" * W)

# ── Training times ──────────────────────────────────────────
train_stages = [
    ("PHATE fit (Stage 1)",            phate_result['train_time_s']),
    ("XGBoost component ranking (S2)", shap_result['xgb_rank_train_time_s']),
    ("SHAP computation (Stage 2)",     shap_result['shap_time_s']),
    ("MLP component predictor (S3)",   mlp_result['train_time_s']),
    ("XGBoost grade clf (Stage 4)",    xgb_final_result['xgb_train_time_s']),
    ("XGBoost TP53 clf (Stage 5b)",    tp53_result['tp53_train_time_s']),
]
total_train = sum(t for _, t in train_stages)

print("\n  ── TRAINING TIME ──")
for name, t in train_stages:
    print(f"  {name:<42} {t:>8.2f} s")
print(f"  {'─' * 52}")
print(f"  {'TOTAL training time':<42} {total_train:>8.2f} s  ({total_train/60:.1f} min)")

# ── Inference times (per sample) ────────────────────────────
infer_stages = [
    ("PHATE transform",       phate_result['infer_time_per_sample_ms']),
    ("MLP forward pass",      xgb_final_result['mlp_infer_per_sample_ms']),
    ("XGBoost grade predict", xgb_final_result['xgb_infer_per_sample_ms']),
    ("XGBoost TP53 predict",  tp53_result['tp53_infer_per_sample_ms']),
]
total_infer = sum(t for _, t in infer_stages)

print("\n  ── INFERENCE TIME (per sample) ──")
for name, t in infer_stages:
    print(f"  {name:<42} {t:>8.3f} ms")
print(f"  {'─' * 52}")
print(f"  {'TOTAL inference time / sample':<42} {total_infer:>8.3f} ms")
print("=" * W)
